# Loan Default Risk Analysis
**Role:** Data Analyst at a consumer lending company

**Goal:** Identify key risk factors driving loan defaults and inform underwriting thresholds

**Dataset:** 500 borrower profiles + 601 loan applications (2024-2025)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

borrowers = pd.read_csv('borrower_profiles.csv')
loans     = pd.read_csv('loan_applications.csv')
df = loans.merge(borrowers, on='borrower_id', how='left')
print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["defaulted"].mean()*100:.1f}%')
df.head(3)

## 1. Overall Default Rate

In [ ]:
default_rate = df['defaulted'].mean() * 100
print(f'Overall default rate: {default_rate:.1f}%')
print(f'Target default rate:  12.0%')
print(f'Excess:               +{default_rate - 12:.1f} percentage points')

fig, ax = plt.subplots(figsize=(6, 4))
counts = df['defaulted'].value_counts()
ax.bar(['Non-Default', 'Default'], [counts[0], counts[1]],
       color=['#4a90d9', '#e74c3c'], edgecolor='white', width=0.5)
ax.set_title('Loan Outcome Distribution', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Loans')
for i, v in enumerate([counts[0], counts[1]]):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('default_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Default Rate by Credit Score Bucket

In [ ]:
bins   = [0, 519, 599, 649, 699, 749, 900]
labels = ['<520', '520-599', '600-649', '650-699', '700-749', '750+']
df['credit_bucket'] = pd.cut(df['credit_score'], bins=bins, labels=labels)

credit_stats = df.groupby('credit_bucket', observed=True).agg(
    loans=('defaulted', 'count'),
    default_rate=('defaulted', 'mean')
).reset_index()
credit_stats['default_rate'] *= 100
print(credit_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#c0392b','#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
bars = ax.bar(credit_stats['credit_bucket'], credit_stats['default_rate'],
              color=colors, edgecolor='white')
ax.axhline(12, color='navy', linestyle='--', linewidth=1.5, label='Target (12%)')
ax.set_xlabel('Credit Score Range')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by Credit Score Bucket', fontweight='bold', fontsize=13)
ax.legend()
for bar, val in zip(bars, credit_stats['default_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('default_by_credit.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. DTI Ratio vs. Default Risk

In [ ]:
dti_bins   = [0, 20, 30, 40, 50, 60, 100]
dti_labels = ['0-20%', '20-30%', '30-40%', '40-50%', '50-60%', '60%+']
df['dti_bucket'] = pd.cut(df['dti_ratio'], bins=dti_bins, labels=dti_labels)

dti_stats = df.groupby('dti_bucket', observed=True).agg(
    loans=('defaulted', 'count'),
    default_rate=('defaulted', 'mean')
).reset_index()
dti_stats['default_rate'] *= 100
print(dti_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(dti_stats['dti_bucket'], dti_stats['default_rate'],
       color='#4a90d9', edgecolor='white')
ax.axhline(12, color='navy', linestyle='--', linewidth=1.5, label='Target (12%)')
ax.set_xlabel('Debt-to-Income Ratio')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by DTI Ratio', fontweight='bold', fontsize=13)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('default_by_dti.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Loan Purpose & Employment Analysis

In [ ]:
purpose_stats = df.groupby('loan_purpose').agg(
    count=('defaulted','count'),
    default_rate=('defaulted','mean'),
).reset_index().sort_values('default_rate', ascending=False)
purpose_stats['default_rate'] *= 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(purpose_stats['loan_purpose'], purpose_stats['default_rate'],
             color='#e74c3c', edgecolor='white')
axes[0].axvline(12, color='navy', linestyle='--', linewidth=1.5, label='Target')
axes[0].set_title('Default Rate by Loan Purpose', fontweight='bold')
axes[0].set_xlabel('Default Rate (%)')
axes[0].legend()

emp_stats = df.groupby('employment_status').agg(
    default_rate=('defaulted','mean')
).reset_index().sort_values('default_rate', ascending=False)
emp_stats['default_rate'] *= 100

axes[1].bar(emp_stats['employment_status'], emp_stats['default_rate'],
            color='#8e44ad', edgecolor='white')
axes[1].axhline(12, color='navy', linestyle='--', linewidth=1.5, label='Target')
axes[1].set_title('Default Rate by Employment Status', fontweight='bold')
axes[1].set_ylabel('Default Rate (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend()

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('default_by_purpose_emp.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
numeric_features = ['credit_score','dti_ratio','loan_amount',
                     'interest_rate','annual_income','years_employed','defaulted']
corr = df[numeric_features].corr()['defaulted'].drop('defaulted').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v > 0 else '#4a90d9' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation with Default (Pearson)', fontweight='bold', fontsize=13)
ax.set_xlabel('Correlation Coefficient')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key correlations:')
print(corr)

## 6. Recommendations for Underwriting Team

In [ ]:
dti_analysis = df.groupby('dti_bucket', observed=True)['defaulted'].mean() * 100
high_dti = dti_analysis[dti_analysis > 25].index.tolist()

credit_analysis = df.groupby('credit_bucket', observed=True)['defaulted'].mean() * 100
high_risk_credit = credit_analysis[credit_analysis > 20].index.tolist()

print('UNDERWRITING RECOMMENDATIONS')
print('=' * 45)
print()
print('RISK FACTOR 1 - Credit Score')
print('  Recommended minimum: 650')
print(f'  High-risk buckets: {high_risk_credit}')
print()
print('RISK FACTOR 2 - Debt-to-Income Ratio')
print('  Recommended maximum DTI: 40%')
print(f'  High-risk DTI buckets (>25% default rate): {high_dti}')
print()
print('RISK FACTOR 3 - Employment Status')
print('  Flag self-employed and unemployed applicants')
print('  for additional income verification.')

## Key Takeaways

- The portfolio default rate significantly exceeds the 12% target.
- **Credit score** is the strongest predictor: borrowers below 650 default at disproportionately high rates.
- **DTI ratio** above 40% is a strong warning signal regardless of credit score.
- **Employment instability** amplifies default probability.
- Implementing recommended thresholds should materially reduce losses without over-restricting access to credit.
